In [ ]:
# 6D POSE ESTIMATION — PER-FRAME ACCURACY (PRED vs GT POSE)
#
# PURPOSE:
#   Run YOLOv5-6D-Pose on a random acc_step sequence, solve PnP independently
#   on both predicted and ground-truth 2D corners, and compare the resulting
#   camera-frame poses to quantify single-frame 6D pose estimation accuracy.
#
# KEY INSIGHT:
#   PnP(GT corners) with a correct camera matrix gives near-zero reprojection
#   error and serves as the "true" camera-frame pose. PnP(pred corners) gives
#   the estimated pose. The difference isolates detector error from PnP error.
#
# PIPELINE (per frame):
#   image → letterbox → YOLOv5-6D → 9 predicted keypoints (pixels)
#   label → denormalize → 9 GT keypoints (pixels)
#   PnP(pred corners) → T_cam_mav_pred (rvec, tvec, R)
#   PnP(GT corners)   → T_cam_mav_gt   (rvec, tvec, R)
#   translation error  = ||tvec_pred − tvec_gt||  [m]
#   rotation error     = arccos((tr(R_pred · R_gt^T) − 1) / 2)  [deg]
#   corner error 2D    = mean L2 over 8 corners (centroid excluded)  [px]
#
# DATA STRUCTURES:
#   OBJECT_POINTS_3D: (9,3) float — mesh 124.ply bounding box in body frame [m]
#                     centroid + 8 corners, extents ≈ 0.47 × 0.47 × 0.38 m
#   CAMERA_MATRIX:    (3,3) float — simulated pinhole, fx=fy=960, cx=960, cy=540
#   DIST_COEFFS:      (5,)  float — all zeros (ideal pinhole, no distortion)
#   pred_pixels:      (9,2) float — [centroid, corner1..8] in original image pixels
#   gt_pixels:        (9,2) float — same layout, from YOLO6D label
#
# INPUT DIRECTORY LAYOUT:
#   data/sequences/acc_step/<seq_name>/drone/
#     images/*.jpg|*.png  — rendered frames (1920×1080)
#     labels/*.txt        — YOLO6D labels (class + 9×2 normalized coords)
#     test.txt            — optional test split (used if present, else last 30%)
#   WEIGHTS: runs/train/unreal/weights/best.pt
#
# DOES:
#   - Picks one random sequence from acc_step/
#   - Loads test split (test.txt) or falls back to last 30% of frames
#   - For each frame: predicts corners, runs PnP on both pred and GT corners
#   - Collects per-frame: translation error, rotation error, 2D corner error,
#     reprojection error (pred + GT), and GT distance (||tvec_gt||)
#   - Prints summary table: mean/median/std for all metrics
#   - Prints distance-binned breakdown (0–2m, 2–4m, 4–6m, 6–10m, 10–50m)
#
# OUTPUT:
#   Console report only — no files saved.
#
# DEPENDENCIES:
#   numpy, opencv, torch, tqdm
#   YOLOv5-6D-Pose repo (models.experimental, utils.{datasets,general,pose_utils,torch_utils})
#
# NOTES:
#   - PnP uses only 8 box corners (indices 1–8); centroid excluded for stability.
#   - Translation error is in camera frame (not world frame); GT distance is
#     ||tvec|| which approximates camera-to-object range.
#   - Rotation error is geodesic (minimum rotation angle between two SO(3) elements).
#   - GT reprojection error ≈ 0.0000 px confirms correct camera matrix + mesh extents.

import numpy as np
import sys
import random
from pathlib import Path
import cv2
import torch
from tqdm import tqdm

sys.path.insert(0, str(Path.home() / 'git/YOLOv5-6D-Pose'))

from models.experimental import attempt_load
from utils.torch_utils import select_device
from utils.datasets import letterbox
from utils.general import check_img_size, scale_coords
from utils.pose_utils import box_filter


# ============================================================================
# 3D MODEL + CAMERA
# ============================================================================

# Mesh extents from 124.ply
min_x, max_x = -0.23410146, 0.23505676
min_y, max_y = -0.23363571, 0.23412720
min_z, max_z = -0.15853004, 0.22499983

OBJECT_POINTS_3D = np.array([
    [(min_x+max_x)/2, (min_y+max_y)/2, (min_z+max_z)/2],
    [min_x, min_y, min_z], [min_x, min_y, max_z],
    [min_x, max_y, min_z], [min_x, max_y, max_z],
    [max_x, min_y, min_z], [max_x, min_y, max_z],
    [max_x, max_y, min_z], [max_x, max_y, max_z],
], dtype=np.float32)

CAMERA_MATRIX = np.array([[960, 0, 960], [0, 960, 540], [0, 0, 1]], dtype=np.float32)
DIST_COEFFS = np.zeros(5, dtype=np.float32)


# ============================================================================
# POSE ESTIMATION
# ============================================================================

def estimate_pose(corners_2d):
    """Run PnP on 8 corners (skip centroid). Returns (success, rvec, tvec, R, reproj_err)."""
    obj_pts = OBJECT_POINTS_3D[1:].astype(np.float32)
    img_pts = corners_2d[1:].astype(np.float32)
    
    success, rvec, tvec = cv2.solvePnP(obj_pts, img_pts, CAMERA_MATRIX, DIST_COEFFS, 
                                         flags=cv2.SOLVEPNP_ITERATIVE)
    if not success:
        return None
    
    R, _ = cv2.Rodrigues(rvec)
    proj, _ = cv2.projectPoints(obj_pts, rvec, tvec, CAMERA_MATRIX, DIST_COEFFS)
    reproj_err = np.mean(np.linalg.norm(proj.reshape(-1, 2) - img_pts, axis=1))
    
    return {'rvec': rvec, 'tvec': tvec.flatten(), 'R': R, 'reproj_err': reproj_err}


def rotation_error_deg(R_pred, R_gt):
    """Geodesic rotation error in degrees."""
    R_diff = R_pred @ R_gt.T
    trace = np.clip((np.trace(R_diff) - 1) / 2, -1, 1)
    return np.degrees(np.arccos(trace))


# ============================================================================
# PREDICTION (same as before)
# ============================================================================

def predict(model, image_path, label_path, img_size=640, conf_thresh=0.01, device='cpu'):
    img0 = cv2.imread(str(image_path))
    if img0 is None:
        return None

    h0, w0 = img0.shape[:2]
    try:
        with open(label_path) as f:
            label_data = [float(x) for x in f.readline().strip().split()]
        if len(label_data) < 19:
            return None
    except:
        return None

    stride = int(model.stride.max())
    img_size = check_img_size(img_size, s=stride)
    img, ratio, pad = letterbox(img0, img_size, stride=stride, auto=False)
    shape = (h0, w0)
    shapes = ((h0, w0), (ratio, pad))

    img_tensor = img[:, :, ::-1].transpose(2, 0, 1)
    img_tensor = np.ascontiguousarray(img_tensor)
    img_tensor = torch.from_numpy(img_tensor).to(device).float() / 255.0
    if img_tensor.ndimension() == 3:
        img_tensor = img_tensor.unsqueeze(0)

    with torch.no_grad():
        pred_out, _ = model(img_tensor)
    pred_out = box_filter(pred_out, conf_thres=conf_thresh, max_det=10)

    if pred_out is None or len(pred_out) == 0 or len(pred_out[0]) == 0:
        return None

    det = pred_out[0][0].clone().cpu()
    confidence = float(det[18])
    corners_pred = det[:18].reshape(1, 18)
    scale_coords(img_tensor.shape[2:], corners_pred, shape, shapes[1])
    pred_pixels = corners_pred[0].numpy().reshape(9, 2)

    gt_norm = np.array([label_data[i] for i in range(1, 19)])
    _, _, height, width = img_tensor.shape
    unpadded_w = width - 2 * pad[0]
    unpadded_h = height - 2 * pad[1]
    gt_letterbox = gt_norm.copy()
    gt_letterbox[::2] = gt_letterbox[::2] * unpadded_w + pad[0]
    gt_letterbox[1::2] = gt_letterbox[1::2] * unpadded_h + pad[1]
    gt_letterbox_tensor = torch.from_numpy(gt_letterbox).reshape(1, 18).float()
    scale_coords(img_tensor.shape[2:], gt_letterbox_tensor, shape, shapes[1])
    gt_pixels = gt_letterbox_tensor[0].numpy().reshape(9, 2)

    return {'pred': pred_pixels, 'gt': gt_pixels, 'confidence': confidence}


# ============================================================================
# MAIN
# ============================================================================

BASE_DIR = Path.home() / 'Desktop/YOLOv5-6D-Pose'
WEIGHTS = str(BASE_DIR / 'runs/train/unreal/weights/best.pt')
SEQ_ROOT = BASE_DIR / 'data/sequences/acc_step'

all_seqs = sorted([d for d in SEQ_ROOT.iterdir() if d.is_dir()])
chosen_seq = random.choice(all_seqs)
seq_dir = chosen_seq / 'drone'
images_dir = seq_dir / 'images'
labels_dir = seq_dir / 'labels'

print(f"Sequence: {chosen_seq.name}")

# Use test split if available
test_txt = seq_dir / 'test.txt'
if test_txt.exists():
    with open(test_txt) as f:
        test_stems = [line.strip() for line in f if line.strip()]
    image_files = []
    for stem in sorted(test_stems):
        p = Path(stem)
        for ext in ['.jpg', '.png']:
            c = images_dir / (p.stem + ext)
            if c.exists():
                image_files.append(c)
                break
    if not image_files:
        image_files = sorted(list(images_dir.glob('*.jpg')) + list(images_dir.glob('*.png')))
    print(f"Test images: {len(image_files)} (from test.txt)")
else:
    image_files = sorted(list(images_dir.glob('*.jpg')) + list(images_dir.glob('*.png')))
    start = int(len(image_files) * 0.70)
    image_files = image_files[start:]
    print(f"Test images: {len(image_files)} (last 30%)")

device = select_device('')
model = attempt_load(WEIGHTS, map_location=device)
model.eval()
print(f"Model loaded\n")

# Per-frame results
trans_errors = []
rot_errors = []
reproj_pred = []
reproj_gt = []
corner_errors_2d = []
gt_distances = []

failed = 0
for img_path in tqdm(image_files, desc="Evaluating"):
    label_path = labels_dir / (img_path.stem + '.txt')
    if not label_path.exists():
        failed += 1
        continue

    res = predict(model, img_path, label_path, device=device)
    if res is None:
        failed += 1
        continue

    pose_gt = estimate_pose(res['gt'])
    pose_pred = estimate_pose(res['pred'])

    if pose_gt is None or pose_pred is None:
        failed += 1
        continue

    # Per-frame errors
    t_err = np.linalg.norm(pose_pred['tvec'] - pose_gt['tvec'])
    r_err = rotation_error_deg(pose_pred['R'], pose_gt['R'])
    corner_err_2d = np.mean(np.linalg.norm(res['pred'][1:] - res['gt'][1:], axis=1))
    gt_dist = np.linalg.norm(pose_gt['tvec'])

    trans_errors.append(t_err)
    rot_errors.append(r_err)
    reproj_pred.append(pose_pred['reproj_err'])
    reproj_gt.append(pose_gt['reproj_err'])
    corner_errors_2d.append(corner_err_2d)
    gt_distances.append(gt_dist)

# Convert
trans_errors = np.array(trans_errors)
rot_errors = np.array(rot_errors)
reproj_pred = np.array(reproj_pred)
reproj_gt = np.array(reproj_gt)
corner_errors_2d = np.array(corner_errors_2d)
gt_distances = np.array(gt_distances)

# ============================================================================
# REPORT
# ============================================================================

print(f"\n{'='*80}")
print(f"6D POSE ACCURACY — {chosen_seq.name}")
print(f"{'='*80}")
print(f"Frames: {len(trans_errors)} successful / {len(image_files)} total / {failed} failed")
print(f"GT distance range: {np.min(gt_distances):.2f} – {np.max(gt_distances):.2f} m")

print(f"\n--- 2D Keypoint Error (pixels) ---")
print(f"  Mean: {np.mean(corner_errors_2d):.2f}  Median: {np.median(corner_errors_2d):.2f}  Std: {np.std(corner_errors_2d):.2f}")

print(f"\n--- Translation Error (meters) ---")
print(f"  Mean: {np.mean(trans_errors):.4f}  Median: {np.median(trans_errors):.4f}  Std: {np.std(trans_errors):.4f}")
print(f"  Min:  {np.min(trans_errors):.4f}  Max: {np.max(trans_errors):.4f}")
print(f"  Relative (% of GT dist): {np.mean(trans_errors / gt_distances) * 100:.2f}%")

print(f"\n--- Rotation Error (degrees) ---")
print(f"  Mean: {np.mean(rot_errors):.2f}  Median: {np.median(rot_errors):.2f}  Std: {np.std(rot_errors):.2f}")
print(f"  Min:  {np.min(rot_errors):.2f}  Max: {np.max(rot_errors):.2f}")

print(f"\n--- Reprojection Error (pixels) ---")
print(f"  Pred corners: {np.mean(reproj_pred):.2f} ± {np.std(reproj_pred):.2f}")
print(f"  GT corners:   {np.mean(reproj_gt):.4f} ± {np.std(reproj_gt):.4f}")

# Breakdown by distance
print(f"\n--- Error vs Distance ---")
for lo, hi in [(0, 2), (2, 4), (4, 6), (6, 10), (10, 50)]:
    mask = (gt_distances >= lo) & (gt_distances < hi)
    n = mask.sum()
    if n > 0:
        print(f"  {lo}-{hi}m ({n} frames): trans={np.mean(trans_errors[mask]):.4f}m  rot={np.mean(rot_errors[mask]):.2f}°  2d={np.mean(corner_errors_2d[mask]):.2f}px")

print(f"{'='*80}")

In [ ]:
# YOLO6D SINGLE-FRAME VISUALIZER — PREDICTED vs GT 3D BBOX OVERLAY
#
# PURPOSE:
#   Run YOLOv5-6D on one frame from step_a3_cam0_1, overlay predicted (red)
#   and ground-truth (green) 3D bounding box wireframes, save annotated PNG.
#
# PIPELINE:
#   image → letterbox → YOLOv5-6D → 9 keypoints (pred, pixels)
#   label → denormalize + undo letterbox → 9 keypoints (GT, pixels)
#   both → 12-edge wireframe + corner markers → matplotlib figure → PNG
#
# CONSTANTS:
#   WEIGHTS:  runs/train/unreal/weights/best.pt
#   SEQUENCE: data/sequences/acc_step/step_a3_cam0_1/drone/
#   labels:   class + 9×2 normalized coords (centroid + 8 corners)
#   edges:    12 pairs defining 3D bbox wireframe (0-based into 8 corners)
#
# OUTPUT:
#   first_frame_debug.png + console stats (confidence, corner error, centroid diff)

import numpy as np
import sys
from pathlib import Path
import cv2
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.home() / "git/YOLOv5-6D-Pose"))

from models.experimental import attempt_load
from utils.torch_utils import select_device
from utils.datasets import letterbox
from utils.general import check_img_size, scale_coords
from utils.pose_utils import box_filter

# ============================================================================
# SETUP
# ============================================================================

BASE_DIR = Path.home() / "Desktop/YOLOv5-6D-Pose"
SEQ_DIR = BASE_DIR / "data/sequences/acc_step/step_a3_cam0_1/drone"
WEIGHTS = str(BASE_DIR / "runs/train/unreal/weights/best.pt")

# Get random image
image_files = sorted((SEQ_DIR / "images").glob("*.jpg"))
if not image_files:
    image_files = sorted((SEQ_DIR / "images").glob("*.png"))
image_path = np.random.choice(image_files)

print(f"Processing: {image_path}")

# Find corresponding label
label_path = SEQ_DIR / "labels" / image_path.with_suffix(".txt").name

print(f"Label: {label_path}")
print(f"Exists: {label_path.exists()}")

if not label_path.exists():
    print("ERROR: Label not found!")
    sys.exit(1)

# ============================================================================
# LOAD GT CORNERS FROM LABEL
# ============================================================================

with open(label_path) as f:
    label_data = [float(x) for x in f.readline().strip().split()]

print(f"Label has {len(label_data)} values")
print(f"Label format: class + 9 keypoints (centroid + 8 corners), normalized 0-1")

# Extract GT corners (indices 1-18: centroid + 8 corners, normalized)
gt_norm = np.array([label_data[i] for i in range(1, 19)])

# ============================================================================
# LOAD IMAGE AND GET PREDICTION
# ============================================================================

img0 = cv2.imread(str(image_path))
h0, w0 = img0.shape[:2]
print(f"Image size: {w0}x{h0}")

device = select_device("cpu")
model = attempt_load(WEIGHTS, map_location=device)
model.eval()

stride = int(model.stride.max())
img_size = check_img_size(640, s=stride)

img, ratio, pad = letterbox(img0, img_size, stride=stride, auto=False)
shape = (h0, w0)
shapes = ((h0, w0), (ratio, pad))

print(f"Letterbox: {img.shape[1]}x{img.shape[0]}, ratio={ratio}, pad={pad}")

img_tensor = img[:, :, ::-1].transpose(2, 0, 1)
img_tensor = np.ascontiguousarray(img_tensor)
img_tensor = torch.from_numpy(img_tensor).to(device).float() / 255.0
if img_tensor.ndimension() == 3:
    img_tensor = img_tensor.unsqueeze(0)

with torch.no_grad():
    pred, _ = model(img_tensor)

pred = box_filter(pred, conf_thres=0.01, max_det=10)

if pred is None or len(pred) == 0 or len(pred[0]) == 0:
    print("ERROR: No detections!")
    sys.exit(1)

det = pred[0][0].clone().cpu()
confidence = float(det[18])
corners_pred = det[:18].reshape(1, 18)

scale_coords(img_tensor.shape[2:], corners_pred, shape, shapes[1])
pred_pixels = corners_pred[0].numpy().reshape(9, 2)

print(f"Prediction successful (confidence: {confidence:.3f})")

# ============================================================================
# SCALE GT CORNERS TO ORIGINAL IMAGE SPACE
# ============================================================================

_, _, height, width = img_tensor.shape

unpadded_w = width - 2 * pad[0]
unpadded_h = height - 2 * pad[1]

gt_letterbox = gt_norm.copy()
gt_letterbox[::2] = gt_letterbox[::2] * unpadded_w
gt_letterbox[1::2] = gt_letterbox[1::2] * unpadded_h

gt_letterbox[::2] += pad[0]
gt_letterbox[1::2] += pad[1]

gt_letterbox_tensor = torch.from_numpy(gt_letterbox).reshape(1, 18).float()
scale_coords(img_tensor.shape[2:], gt_letterbox_tensor, shape, shapes[1])

gt_pixels = gt_letterbox_tensor[0].numpy().reshape(9, 2)

# ============================================================================
# VISUALIZE
# ============================================================================

edges = [[0,1],[0,2],[0,4],[1,3],[1,5],[2,3],[2,6],[3,7],[4,5],[4,6],[5,7],[6,7]]

fig, ax = plt.subplots(1, 1, figsize=(12, 8))

img_rgb = cv2.cvtColor(img0, cv2.COLOR_BGR2RGB)
ax.imshow(img_rgb)

# Draw GT (green)
for i, edge in enumerate(edges):
    ax.plot([gt_pixels[edge[0]+1, 0], gt_pixels[edge[1]+1, 0]],
            [gt_pixels[edge[0]+1, 1], gt_pixels[edge[1]+1, 1]],
            "g-", linewidth=3, label="Ground Truth" if i==0 else "")
ax.scatter(gt_pixels[1:, 0], gt_pixels[1:, 1], c="lime", s=100,
          edgecolors="white", linewidths=2, zorder=5)
ax.scatter(gt_pixels[0, 0], gt_pixels[0, 1], c="lime", s=200,
          marker="*", edgecolors="black", linewidths=3, zorder=5)

# Draw Pred (red)
for i, edge in enumerate(edges):
    ax.plot([pred_pixels[edge[0]+1, 0], pred_pixels[edge[1]+1, 0]],
            [pred_pixels[edge[0]+1, 1], pred_pixels[edge[1]+1, 1]],
            "r-", linewidth=2, linestyle="--", label="Predicted" if i==0 else "")
ax.scatter(pred_pixels[1:, 0], pred_pixels[1:, 1], c="red", s=80,
          marker="x", linewidths=3, zorder=5)
ax.scatter(pred_pixels[0, 0], pred_pixels[0, 1], c="orange", s=150,
          marker="*", linewidths=3, zorder=5)

corner_error = np.mean(np.linalg.norm(pred_pixels[1:] - gt_pixels[1:], axis=1))

ax.set_title(f"Frame: {image_path.name} | Confidence: {confidence:.3f} | Corner Error: {corner_error:.2f}px",
            fontsize=14, fontweight="bold")
ax.legend(fontsize=12)
ax.axis("off")

plt.tight_layout()
plt.savefig("first_frame_debug.png", dpi=150, bbox_inches="tight")
print(f"Saved: first_frame_debug.png")
print(f"Corner error: {corner_error:.2f} pixels")
print(f"Centroid GT:   {gt_pixels[0]}")
print(f"Centroid Pred: {pred_pixels[0]}")
print(f"Centroid diff: {np.linalg.norm(pred_pixels[0] - gt_pixels[0]):.2f} pixels")

plt.show()

In [ ]:
# YOLO6D SEQUENCE → VIDEO — PREDICTED vs GT 3D BBOX OVERLAY
#
# PURPOSE:
#   Run YOLOv5-6D on every frame of a random acc_step sequence, draw predicted
#   (red) and GT (green) 3D bbox wireframes, encode as MP4 video.
#
# PIPELINE (per frame):
#   image → letterbox → YOLOv5-6D → 9 pred keypoints (pixels)
#   label → denormalize + undo letterbox → 9 GT keypoints (pixels)
#   both → 12-edge wireframe + corner dots + text HUD → VideoWriter
#
# CONSTANTS:
#   WEIGHTS:  runs/train/unreal/weights/best.pt
#   SEQUENCE: random from data/sequences/acc_step/
#   edges:    12 pairs defining 3D bbox wireframe (0-based, +1 to skip centroid)
#   codec:    mp4v → XVID fallback, 30 fps
#
# OUTPUT:
#   sequence_{name}.mp4 — annotated video at original resolution (1920×1080)
#   Console: sequence name, frame count, resolution, file size

import numpy as np
import sys
from pathlib import Path
import cv2
import torch
import random
from tqdm import tqdm

sys.path.insert(0, str(Path.home() / 'git/YOLOv5-6D-Pose'))

from models.experimental import attempt_load
from utils.torch_utils import select_device
from utils.datasets import letterbox
from utils.general import check_img_size, scale_coords
from utils.pose_utils import box_filter

# ============================================================================
# SETUP
# ============================================================================

BASE_DIR = Path.home() / 'Desktop/YOLOv5-6D-Pose'
WEIGHTS = str(BASE_DIR / 'runs/train/unreal/weights/best.pt')
SEQ_ROOT = BASE_DIR / 'data/sequences/acc_step'

# Pick a random sequence
all_seqs = sorted([d for d in SEQ_ROOT.iterdir() if d.is_dir()])
chosen_seq = random.choice(all_seqs)
seq_dir = chosen_seq / 'drone'
images_dir = seq_dir / 'images'
labels_dir = seq_dir / 'labels'

OUTPUT_VIDEO = f'sequence_{chosen_seq.name}.mp4'

print(f"Sequence: {chosen_seq.name}")

# Get all images
image_files = sorted(list(images_dir.glob('*.jpg')) + list(images_dir.glob('*.png')))
print(f"Found {len(image_files)} images")

# Load model
device = select_device('cpu')
model = attempt_load(WEIGHTS, map_location=device)
model.eval()
print("Model loaded\n")

# 3D bounding box edges (0-based into 8 corners; +1 when indexing 9x2 arrays)
edges = [[0,1],[0,2],[0,4],[1,3],[1,5],[2,3],[2,6],[3,7],[4,5],[4,6],[5,7],[6,7]]

# Get first image to determine video size
first_img = cv2.imread(str(image_files[0]))
h0, w0 = first_img.shape[:2]

# Setup video writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, 30.0, (w0, h0))

if not out.isOpened():
    print("mp4v failed, trying XVID...")
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    OUTPUT_VIDEO = f'sequence_{chosen_seq.name}.avi'
    out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, 30.0, (w0, h0))
    if not out.isOpened():
        print("ERROR: Video writer failed!")
        sys.exit(1)

print(f"Video writer: {w0}x{h0} @ 30fps")

stride = int(model.stride.max())
img_size = check_img_size(640, s=stride)

# Process all frames
frames_written = 0
for img_path in tqdm(image_files, desc="Creating video"):

    label_path = labels_dir / (img_path.stem + '.txt')
    if not label_path.exists():
        continue

    # Load GT corners
    with open(label_path) as f:
        label_data = [float(x) for x in f.readline().strip().split()]

    if len(label_data) < 19:
        continue

    gt_norm = np.array([label_data[i] for i in range(1, 19)])

    # Load image
    img0 = cv2.imread(str(img_path))
    if img0 is None:
        continue

    h0, w0 = img0.shape[:2]

    # Prepare image for model
    img, ratio, pad = letterbox(img0, img_size, stride=stride, auto=False)
    shape = (h0, w0)
    shapes = ((h0, w0), (ratio, pad))

    img_tensor = img[:, :, ::-1].transpose(2, 0, 1)
    img_tensor = np.ascontiguousarray(img_tensor)
    img_tensor = torch.from_numpy(img_tensor).to(device).float() / 255.0
    if img_tensor.ndimension() == 3:
        img_tensor = img_tensor.unsqueeze(0)

    with torch.no_grad():
        pred, _ = model(img_tensor)

    pred = box_filter(pred, conf_thres=0.01, max_det=10)

    if pred is None or len(pred) == 0 or len(pred[0]) == 0:
        continue

    det = pred[0][0].clone().cpu()
    confidence = float(det[18])
    corners_pred = det[:18].reshape(1, 18)

    scale_coords(img_tensor.shape[2:], corners_pred, shape, shapes[1])
    pred_pixels = corners_pred[0].numpy().reshape(9, 2)

    # Scale GT corners to original image space
    _, _, height, width = img_tensor.shape
    unpadded_w = width - 2 * pad[0]
    unpadded_h = height - 2 * pad[1]

    gt_letterbox = gt_norm.copy()
    gt_letterbox[::2] = gt_letterbox[::2] * unpadded_w + pad[0]
    gt_letterbox[1::2] = gt_letterbox[1::2] * unpadded_h + pad[1]

    gt_letterbox_tensor = torch.from_numpy(gt_letterbox).reshape(1, 18).float()
    scale_coords(img_tensor.shape[2:], gt_letterbox_tensor, shape, shapes[1])
    gt_pixels = gt_letterbox_tensor[0].numpy().reshape(9, 2)

    # Compute error
    corner_error = np.mean(np.linalg.norm(pred_pixels[1:] - gt_pixels[1:], axis=1))

    # Draw on frame
    frame = img0.copy()

    # GT (green)
    for edge in edges:
        pt1 = (int(gt_pixels[edge[0]+1, 0]), int(gt_pixels[edge[0]+1, 1]))
        pt2 = (int(gt_pixels[edge[1]+1, 0]), int(gt_pixels[edge[1]+1, 1]))
        cv2.line(frame, pt1, pt2, (0, 255, 0), 1)
    for i in range(1, 9):
        pt = (int(gt_pixels[i, 0]), int(gt_pixels[i, 1]))
        cv2.circle(frame, pt, 4, (0, 255, 0), -1)

    # Pred (red)
    for edge in edges:
        pt1 = (int(pred_pixels[edge[0]+1, 0]), int(pred_pixels[edge[0]+1, 1]))
        pt2 = (int(pred_pixels[edge[1]+1, 0]), int(pred_pixels[edge[1]+1, 1]))
        cv2.line(frame, pt1, pt2, (0, 0, 255), 1)
    for i in range(1, 9):
        pt = (int(pred_pixels[i, 0]), int(pred_pixels[i, 1]))
        cv2.circle(frame, pt, 3, (0, 0, 255), -1)

    # Text overlay
    cv2.putText(frame, f"Frame: {img_path.name}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    cv2.putText(frame, f"Confidence: {confidence:.3f}", (10, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    cv2.putText(frame, f"Error: {corner_error:.2f}px", (10, 90),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    cv2.putText(frame, "Green: GT | Red: Pred", (10, h0-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    out.write(frame)
    frames_written += 1

out.release()

print(f"\nVideo saved: {OUTPUT_VIDEO}")
print(f"  Sequence: {chosen_seq.name}")
print(f"  Total images: {len(image_files)}")
print(f"  Frames written: {frames_written}")
print(f"  Resolution: {w0}x{h0}")

if frames_written > 0:
    video_path = Path(OUTPUT_VIDEO)
    if video_path.exists():
        size_mb = video_path.stat().st_size / (1024 * 1024)
        print(f"  File size: {size_mb:.2f} MB")
else:
    print("\nWARNING: No frames written!")

In [ ]:
#!/usr/bin/env python3
# LINK RENDERED IMAGE FOLDERS → DATASET JSON ENTRIES
#
# PURPOSE:
#   Match acc_step image folders (step_a{level}_cam{cam_id}_{suffix}) to
#   dataset_acc_step.json entries by (level, cam_id) pair, add "image_dir"
#   field to each JSON entry so downstream scripts can find rendered frames.
#
# PIPELINE:
#   scan acc_step/ folders → parse level + cam_id from name via regex
#   load JSON → build (level, cam_id) → key lookup
#   match pairs → print mapping table → write "image_dir" into JSON
#
# INPUT:
#   data/sequences/acc_step/step_a{L}_cam{C}_{S}/  — 35 rendered folders
#   dataset_acc_step.json — 35 entries keyed "0"–"34" with level + cam_id
#
# OUTPUT:
#   Updated dataset_acc_step.json with "image_dir" field added per entry.
#   Console: mapping table + match/unmatch report.

import json
import re
from pathlib import Path

BASE_DIR = Path.home() / "Desktop/YOLOv5-6D-Pose/data/sequences/acc_step"
JSON_PATH = Path.home() / "Desktop/rl_tracking/examples/deep_eagle/dataset_acc_step.json"

# ── Load JSON ──
with open(JSON_PATH) as f:
    dataset = json.load(f)

# Build lookup: (level, cam_id) → json_key
json_lookup = {}
for key, entry in dataset.items():
    pair = (entry["level"], entry["cam_id"])
    json_lookup[pair] = key

# ── Scan image folders ──
folder_pattern = re.compile(r"step_a(\d+)_cam(\d+)_(\d+)")

matches = []
unmatched_folders = []
unmatched_json = set(json_lookup.keys())

for folder in sorted(BASE_DIR.iterdir()):
    if not folder.is_dir():
        continue
    m = folder_pattern.match(folder.name)
    if not m:
        continue

    level = int(m.group(1))
    cam_id = int(m.group(2))
    suffix = int(m.group(3))
    pair = (level, cam_id)

    if pair in json_lookup:
        json_key = json_lookup[pair]
        n_images = len(list((folder / "drone" / "images").glob("*"))) if (folder / "drone" / "images").exists() else 0
        matches.append({
            "folder": folder.name,
            "json_key": json_key,
            "level": level,
            "cam_id": cam_id,
            "suffix": suffix,
            "n_images": n_images,
        })
        unmatched_json.discard(pair)
    else:
        unmatched_folders.append(folder.name)

# ── Print table ──
print(f"{'json_key':>8}  {'level':>5}  {'cam':>3}  {'folder':<30}  {'suffix':>6}  {'images':>6}")
print("-" * 70)
for m in sorted(matches, key=lambda x: int(x["json_key"])):
    print(f"{m['json_key']:>8}  {m['level']:>5}  {m['cam_id']:>3}  {m['folder']:<30}  {m['suffix']:>6}  {m['n_images']:>6}")

print(f"\nMatched: {len(matches)} / {len(dataset)}")

if unmatched_folders:
    print(f"\nUnmatched folders ({len(unmatched_folders)}):")
    for f in unmatched_folders:
        print(f"  {f}")

if unmatched_json:
    print(f"\nUnmatched JSON entries ({len(unmatched_json)}):")
    for pair in sorted(unmatched_json):
        key = json_lookup[pair]
        print(f"  key={key}  level={pair[0]}  cam={pair[1]}")

# ── Update JSON with image_dir ──
for m in matches:
    dataset[m["json_key"]]["image_dir"] = m["folder"]

# Save updated JSON
with open(JSON_PATH, "w") as f:
    json.dump(dataset, f, indent=2)
print(f"\nUpdated JSON saved: {JSON_PATH}")

In [8]:
# PnP RESULTS JSON — INSPECT RANDOM SEQUENCE KEYS
#
# PURPOSE:
#   Load pnp_results.json, pick one random sequence, print all keys
#   with types, lengths, and first-element samples for quick sanity check.
#
# INPUT:
#   estimation_data/unreal/acc_step/pnp_results.json (35 sequences)
#
# OUTPUT:
#   Console: key name, type/length, first value preview per field.

import json, random

with open("estimation_data/unreal/acc_step/pnp_results.json") as f:
    data = json.load(f)

key = random.choice(list(data.keys()))
print(f"Sequence key: {key}\n")
for k, v in data[key].items():
    if isinstance(v, list):
        print(f"  {k:<20} list[{len(v)}]  sample: {str(v[0])[:80]}")
    else:
        print(f"  {k:<20} {type(v).__name__}: {v}")

Sequence key: 31

  orig_session         str: sim_acc_step
  orig_sequence        str: step_a21_cam1
  level                int: 21
  cam_id               int: 1
  t                    list[150]  sample: 0.0
  pos                  list[150]  sample: [-0.04560290508428744, -2.016060743794103, 1.0533059727539615]
  vel                  list[150]  sample: [-2.5835379069370777e-16, -3.355518028488625e-16, 3.860427572975443e-07]
  quat                 list[150]  sample: [7.926341314970554e-18, -7.580792059795039e-18, 6.422675681513454e-17, 1.0]
  omega_b              list[150]  sample: [1.3010584905594845e-16, 1.1713328862427935e-16, 1.0866793945764759e-16]
  rts_pos_s            list[150]  sample: [-0.04560290508428744, -2.016060743794103, 1.0533059727539615]
  rts_vel              list[150]  sample: [-2.5835379069370777e-16, -3.355518028488625e-16, 3.860427572975443e-07]
  rts_acc              list[150]  sample: [-1.225676259939548e-16, -1.1374452147382392e-16, -7.838485421111409e-07]
  f

In [ ]:
# YOLO6D PnP RESULTS — PER-SEQUENCE 4×2 DASHBOARD
#
# PURPOSE:
#   Plot a comprehensive 8-panel dashboard for each of 35 acc_step sequences,
#   overlaying PnP-estimated pose on GT and showing per-frame error metrics.
#
# SUBPLOTS (per sequence):
#   (0,0) Position:         GT xyz (solid) + PnP xyz (dots) [m]
#   (0,1) Orientation:      GT RPY (solid) + PnP RPY (dots) [deg]
#   (1,0) Position error:   ||pnp − gt|| [mm] + mean line
#   (1,1) Angular error:    geodesic 2·arccos(|q·q|) [deg] + mean line
#   (2,0) Velocity:         GT vel xyz [m/s]
#   (2,1) Acceleration:     fd_acc xyz + aref dashed [m/s²]
#   (3,0) Angular velocity: ωx/ωy/ωz body-frame [deg/s]
#   (3,1) Per-axis error:   signed Δx/Δy/Δz [mm]
#
# INPUT:
#   pnp_results.json — 35 sequences × 150 frames with GT + PnP poses
#
# OUTPUT:
#   35 inline matplotlib figures (not saved).

import json, numpy as np, matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation as Rot

path = "/home/michal/Desktop/rl_tracking/examples/deep_eagle/estimation_data/unreal/acc_step/pnp_results.json"
with open(path) as f:
    data = json.load(f)

def quat_to_rpy(q):
    """xyzw quaternion array (N,4) → RPY degrees (N,3)"""
    x, y, z, w = q[:,0], q[:,1], q[:,2], q[:,3]
    n = np.sqrt(x*x+y*y+z*z+w*w); n = np.where(n>0, n, 1.)
    x, y, z, w = x/n, y/n, z/n, w/n
    roll  = np.arctan2(2*(w*x+y*z), 1-2*(x*x+y*y))
    pitch = np.arcsin(np.clip(2*(w*y-z*x), -1, 1))
    yaw   = np.arctan2(2*(w*z+x*y), 1-2*(y*y+z*z))
    return np.rad2deg(np.stack([roll, pitch, yaw], axis=1))

def angular_error_deg(q_est, q_gt):
    """Per-frame geodesic angular error in degrees."""
    dot = np.sum(q_est * q_gt, axis=1)
    dot = np.clip(np.abs(dot), 0, 1)
    return np.rad2deg(2 * np.arccos(dot))

C = ["tab:red", "tab:green", "tab:blue"]
XYZ = ["x", "y", "z"]
RPY = ["roll", "pitch", "yaw"]

for seq_key in sorted(data.keys(), key=int):
    e = data[seq_key]
    t = np.array(e["t"])
    N = len(t)

    gt_pos = np.array(e["pos"])          # (N,3)
    gt_vel = np.array(e["vel"])          # (N,3)
    gt_quat = np.array(e["quat"])        # (N,4) xyzw
    pnp_pos = np.array(e["pnp_pos"])     # (N,3)
    pnp_q = np.array(e["pnp_q"])        # (N,4) xyzw
    aref = np.array(e["aref"])           # (N,)
    acc = np.array(e.get("rts_acc", e.get("fd_acc", [[0,0,0]]*N)))
    omega = np.array(e.get("omega_b", [[0,0,0]]*N))

    # Masks
    pnp_ok = ~np.any(np.isnan(pnp_pos), axis=1)
    gt_rpy = quat_to_rpy(gt_quat)

    # PnP RPY (only valid frames)
    pnp_rpy = np.full((N, 3), np.nan)
    if pnp_ok.any():
        pnp_rpy[pnp_ok] = quat_to_rpy(pnp_q[pnp_ok])

    # Errors
    pos_err = np.full(N, np.nan)
    ang_err = np.full(N, np.nan)
    if pnp_ok.any():
        pos_err[pnp_ok] = np.linalg.norm(pnp_pos[pnp_ok] - gt_pos[pnp_ok], axis=1)
        ang_err[pnp_ok] = angular_error_deg(pnp_q[pnp_ok], gt_quat[pnp_ok])

    tag = f"{e.get('orig_sequence', seq_key)} | level={e['level']} cam={e['cam_id']} | {pnp_ok.sum()}/{N} OK"

    fig, axes = plt.subplots(4, 2, figsize=(18, 18))

    # ── (0,0) Position ──
    ax = axes[0, 0]
    for i in range(3):
        ax.plot(t, gt_pos[:, i], color=C[i], lw=1.5, label=f"GT {XYZ[i]}")
        ax.plot(t[pnp_ok], pnp_pos[pnp_ok, i], ".", color=C[i], ms=3, alpha=0.6, label=f"PnP {XYZ[i]}")
    ax.set_ylabel("pos [m]"); ax.set_title("Position"); ax.legend(ncol=3, fontsize=7); ax.grid(True, alpha=0.3)

    # ── (0,1) Orientation ──
    ax = axes[0, 1]
    for i in range(3):
        ax.plot(t, gt_rpy[:, i], color=C[i], lw=1.5, label=f"GT {RPY[i]}")
        ax.plot(t[pnp_ok], pnp_rpy[pnp_ok, i], ".", color=C[i], ms=3, alpha=0.6, label=f"PnP {RPY[i]}")
    ax.set_ylabel("[deg]"); ax.set_title("Orientation (RPY)"); ax.legend(ncol=3, fontsize=7); ax.grid(True, alpha=0.3)

    # ── (1,0) Position error ──
    ax = axes[1, 0]
    ax.plot(t[pnp_ok], pos_err[pnp_ok] * 1000, "b.-", ms=2, alpha=0.7, lw=0.8)
    if pnp_ok.any():
        ax.axhline(np.nanmean(pos_err) * 1000, color="b", ls="--", lw=0.8, alpha=0.5,
                   label=f"mean={np.nanmean(pos_err)*1000:.1f}mm")
    ax.set_ylabel("pos error [mm]"); ax.set_title("Position Error"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # ── (1,1) Angular error ──
    ax = axes[1, 1]
    ax.plot(t[pnp_ok], ang_err[pnp_ok], "r.-", ms=2, alpha=0.7, lw=0.8)
    if pnp_ok.any():
        ax.axhline(np.nanmean(ang_err), color="r", ls="--", lw=0.8, alpha=0.5,
                   label=f"mean={np.nanmean(ang_err):.2f}°")
    ax.set_ylabel("ang error [deg]"); ax.set_title("Angular Error"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # ── (2,0) Velocity ──
    ax = axes[2, 0]
    for i in range(3):
        ax.plot(t, gt_vel[:, i], color=C[i], lw=1.2, label=f"{XYZ[i]}")
    ax.set_ylabel("vel [m/s]"); ax.set_title("Velocity (GT)"); ax.legend(ncol=3, fontsize=7); ax.grid(True, alpha=0.3)

    # ── (2,1) Acceleration + aref ──
    ax = axes[2, 1]
    for i in range(3):
        ax.plot(t, acc[:, i], color=C[i], lw=1.2, label=f"acc {XYZ[i]}")
    ax.plot(t, aref, "k--", lw=1.0, alpha=0.6, label="a_ref")
    ax.set_ylabel("acc [m/s²]"); ax.set_title("Acceleration + Reference"); ax.legend(ncol=4, fontsize=7); ax.grid(True, alpha=0.3)

    # ── (3,0) Angular velocity ──
    ax = axes[3, 0]
    lbl_w = ["ωx", "ωy", "ωz"]
    for i in range(3):
        ax.plot(t, np.rad2deg(np.array(omega)[:, i]), color=C[i], lw=1.2, label=lbl_w[i])
    ax.set_ylabel("ω [deg/s]"); ax.set_xlabel("time [s]"); ax.set_title("Angular Velocity (body)")
    ax.legend(ncol=3, fontsize=7); ax.grid(True, alpha=0.3)

    # ── (3,1) Per-axis position error ──
    ax = axes[3, 1]
    for i in range(3):
        err_i = np.full(N, np.nan)
        if pnp_ok.any():
            err_i[pnp_ok] = (pnp_pos[pnp_ok, i] - gt_pos[pnp_ok, i]) * 1000
        ax.plot(t[pnp_ok], err_i[pnp_ok], ".", color=C[i], ms=3, alpha=0.6, label=f"Δ{XYZ[i]}")
    ax.axhline(0, color="k", lw=0.5, alpha=0.3)
    ax.set_ylabel("error [mm]"); ax.set_xlabel("time [s]"); ax.set_title("Per-Axis Position Error (signed)")
    ax.legend(ncol=3, fontsize=7); ax.grid(True, alpha=0.3)

    fig.suptitle(tag, fontsize=14, fontweight="bold")
    fig.tight_layout()
    plt.show()

In [ ]:
# YOLO6D PnP — AGGREGATE ERROR vs DISTANCE (LOG SCALE)
#
# PURPOSE:
#   Concatenate all 35 acc_step sequences, plot position and orientation
#   error vs camera-to-target distance on log axes, plus error distributions.
#
# SUBPLOTS (3×2, log y-axes):
#   (0,0) Per-axis |Δx|,|Δy|,|Δz| vs distance + binned medians [mm]
#   (0,1) Total ||Δpos|| vs distance + median + 90th percentile [mm]
#   (1,0) Geodesic angular error vs distance + median + 90th pctl [deg]
#   (1,1) |Δyaw| vs distance + median [deg]
#   (2,0) Position error histogram (log-spaced bins) + mean/median
#   (2,1) Angular error histogram (log-spaced bins) + mean/median
#
# INPUT:
#   pnp_results.json — 35 seq × 150 frames, ~5250 total
#
# OUTPUT:
#   Single inline figure + console stats (mean/median pos mm, ang deg).

import json, numpy as np, matplotlib.pyplot as plt

path = "/home/michal/Desktop/rl_tracking/examples/deep_eagle/estimation_data/unreal/acc_step/pnp_results.json"
with open(path) as f:
    data = json.load(f)

# Concatenate all sequences
all_gt_pos, all_pnp_pos, all_gt_q, all_pnp_q, all_cam_pos, all_level = [], [], [], [], [], []

for seq_key in sorted(data.keys(), key=int):
    e = data[seq_key]
    gt_pos = np.array(e["pos"])
    pnp_pos = np.array(e["pnp_pos"])
    gt_q = np.array(e["quat"])
    pnp_q = np.array(e["pnp_q"])
    cam_pos = np.array(e["cam_pos"])
    ok = ~np.any(np.isnan(pnp_pos), axis=1)
    if not ok.any():
        continue
    all_gt_pos.append(gt_pos[ok]);  all_pnp_pos.append(pnp_pos[ok])
    all_gt_q.append(gt_q[ok]);     all_pnp_q.append(pnp_q[ok])
    all_cam_pos.append(cam_pos[ok]); all_level.append(np.full(ok.sum(), e["level"]))

gt_pos = np.concatenate(all_gt_pos);   pnp_pos = np.concatenate(all_pnp_pos)
gt_q = np.concatenate(all_gt_q);       pnp_q = np.concatenate(all_pnp_q)
cam_pos = np.concatenate(all_cam_pos);  levels = np.concatenate(all_level)
distance = np.linalg.norm(cam_pos - gt_pos, axis=1)

pos_err_xyz = (pnp_pos - gt_pos) * 1000  # mm
pos_err_norm = np.linalg.norm(pos_err_xyz, axis=1)

def quat_to_rpy(q):
    x, y, z, w = q[:,0], q[:,1], q[:,2], q[:,3]
    n = np.sqrt(x*x+y*y+z*z+w*w); n = np.where(n>0, n, 1.)
    x, y, z, w = x/n, y/n, z/n, w/n
    roll  = np.arctan2(2*(w*x+y*z), 1-2*(x*x+y*y))
    pitch = np.arcsin(np.clip(2*(w*y-z*x), -1, 1))
    yaw   = np.arctan2(2*(w*z+x*y), 1-2*(y*y+z*z))
    return np.rad2deg(np.stack([roll, pitch, yaw], axis=1))

gt_rpy = quat_to_rpy(gt_q);  pnp_rpy = quat_to_rpy(pnp_q)
rpy_err = (pnp_rpy - gt_rpy + 180) % 360 - 180
dot = np.sum(pnp_q * gt_q, axis=1)
ang_err = np.rad2deg(2 * np.arccos(np.clip(np.abs(dot), 0, 1)))

# Binning
bins = np.linspace(distance.min(), distance.max(), 20)
bin_c = 0.5 * (bins[:-1] + bins[1:])
idx = np.digitize(distance, bins)
valid_bins = [b for b in range(1, len(bins)) if (idx == b).sum() > 0]
bc = [bin_c[b-1] for b in valid_bins]

print(f"Total: {len(distance)} frames | dist {distance.min():.2f}–{distance.max():.2f}m")
print(f"Pos err: mean={pos_err_norm.mean():.1f}mm  median={np.median(pos_err_norm):.1f}mm")
print(f"Ang err: mean={ang_err.mean():.2f}°  median={np.median(ang_err):.2f}°")

C = ["tab:red", "tab:green", "tab:blue"]
XYZ = ["X", "Y", "Z"]

fig, axes = plt.subplots(3, 2, figsize=(18, 16))

# ── (0,0) Per-axis position error vs distance ──
ax = axes[0, 0]
for i in range(3):
    ax.scatter(distance, np.abs(pos_err_xyz[:, i]), s=1, alpha=0.1, color=C[i])
    med = [np.median(np.abs(pos_err_xyz[idx == b, i])) for b in valid_bins]
    ax.plot(bc, med, "-o", color=C[i], lw=2, ms=4, label=f"med |Δ{XYZ[i]}|")
ax.set_yscale("log"); ax.set_xlabel("distance [m]"); ax.set_ylabel("|error| [mm]")
ax.set_title("Per-Axis Position Error vs Distance"); ax.legend(ncol=3, fontsize=7); ax.grid(True, alpha=0.3, which="both")

# ── (0,1) Total position error vs distance ──
ax = axes[0, 1]
ax.scatter(distance, pos_err_norm, s=1, alpha=0.1, color="steelblue")
med = [np.median(pos_err_norm[idx == b]) for b in valid_bins]
p90 = [np.percentile(pos_err_norm[idx == b], 90) for b in valid_bins]
ax.plot(bc, med, "-o", color="darkblue", lw=2, ms=4, label="median")
ax.plot(bc, p90, "--s", color="orange", lw=1.5, ms=3, label="90th pctl")
ax.set_yscale("log"); ax.set_xlabel("distance [m]"); ax.set_ylabel("error [mm]")
ax.set_title("Total Position Error vs Distance"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3, which="both")

# ── (1,0) Angular error vs distance ──
ax = axes[1, 0]
ax.scatter(distance, ang_err, s=1, alpha=0.1, color="indianred")
med = [np.median(ang_err[idx == b]) for b in valid_bins]
p90 = [np.percentile(ang_err[idx == b], 90) for b in valid_bins]
ax.plot(bc, med, "-o", color="darkred", lw=2, ms=4, label="median")
ax.plot(bc, p90, "--s", color="orange", lw=1.5, ms=3, label="90th pctl")
ax.set_yscale("log"); ax.set_xlabel("distance [m]"); ax.set_ylabel("error [deg]")
ax.set_title("Geodesic Angular Error vs Distance"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3, which="both")

# ── (1,1) Yaw error vs distance ──
ax = axes[1, 1]
ax.scatter(distance, np.abs(rpy_err[:, 2]), s=1, alpha=0.1, color="tab:purple")
med = [np.median(np.abs(rpy_err[idx == b, 2])) for b in valid_bins]
ax.plot(bc, med, "-o", color="purple", lw=2, ms=4, label="median |Δyaw|")
ax.set_yscale("log"); ax.set_xlabel("distance [m]"); ax.set_ylabel("|yaw error| [deg]")
ax.set_title("Yaw Error vs Distance"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3, which="both")

# ── (2,0) Position error distribution ──
ax = axes[2, 0]
ax.hist(pos_err_norm, bins=np.logspace(np.log10(max(pos_err_norm.min(), 0.1)), np.log10(pos_err_norm.max()), 60),
        color="steelblue", alpha=0.7, edgecolor="white", lw=0.3)
ax.axvline(np.median(pos_err_norm), color="k", ls="--", lw=1.5, label=f"median={np.median(pos_err_norm):.1f}mm")
ax.axvline(np.mean(pos_err_norm), color="r", ls="--", lw=1.5, label=f"mean={np.mean(pos_err_norm):.1f}mm")
ax.set_xscale("log"); ax.set_xlabel("position error [mm]"); ax.set_ylabel("count")
ax.set_title("Position Error Distribution"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3, which="both")

# ── (2,1) Angular error distribution ──
ax = axes[2, 1]
ax.hist(ang_err, bins=np.logspace(np.log10(max(ang_err.min(), 0.01)), np.log10(ang_err.max()), 60),
        color="indianred", alpha=0.7, edgecolor="white", lw=0.3)
ax.axvline(np.median(ang_err), color="k", ls="--", lw=1.5, label=f"median={np.median(ang_err):.2f}°")
ax.axvline(np.mean(ang_err), color="darkred", ls="--", lw=1.5, label=f"mean={np.mean(ang_err):.2f}°")
ax.set_xscale("log"); ax.set_xlabel("angular error [deg]"); ax.set_ylabel("count")
ax.set_title("Angular Error Distribution"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3, which="both")

fig.suptitle(f"All sequences | {len(distance)} frames | dist {distance.min():.1f}–{distance.max():.1f}m",
             fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()